DAY 3

In [ ]:
%pip install -q langchain "langchain-google-genai<4.5" "google-genai>=2.12.1,<3" "google-auth==2.49.0"

In [ ]:
import os
from getpass import getpass
from google.colab import userdata
from google import genai
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.output_parsers import StrOutputParser

try:
    api_key = userdata.get("GEMINI_API_KEY")
except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
    api_key = getpass("Enter your Gemini API key: ")
api_key = (api_key or "").strip()
if not api_key:
    raise ValueError("Add GEMINI_API_KEY to Colab Secrets and enable notebook access.")
os.environ["GOOGLE_API_KEY"] = api_key

client = genai.Client(api_key=api_key)
models = [m.name.removeprefix("models/") for m in client.models.list()
          if m.name and "gemini" in m.name.lower()
          and "flash" in m.name.lower()
          and "generateContent" in (m.supported_actions or [])
          and not any(x in m.name.lower() for x in
                      ("image", "tts", "audio", "live", "embedding", "robotics"))]
# Prefer lightweight text models, then other available Flash models.
models.sort(key=lambda name: ("lite" not in name, "preview" in name or "exp" in name, name))

def is_temporary_error(exc):
    message = str(exc).upper()
    return any(x in message for x in ("503", "429", "500", "502", "504", "UNAVAILABLE", "RESOURCE_EXHAUSTED", "DEADLINE_EXCEEDED", "TIMEOUT"))

candidates = []
for model in models:
    try:
        candidates.append(ChatGoogleGenerativeAI(model=model, api_key=api_key,
                          temperature=0.3, max_retries=1, timeout=60))
    except Exception as exc:
        print(f"Cannot initialize {model}: {type(exc).__name__}")

llm = None
for candidate in candidates:
    try:
        reply = candidate.invoke("Reply with Hello.")
        if not StrOutputParser().invoke(reply).strip():
            continue
        llm = candidate
        print("Connected successfully:", candidate.model)
        break
    except Exception as exc:
        print(f"Unavailable: {candidate.model} ({type(exc).__name__})")
        if any(x in str(exc).upper() for x in ("API_KEY_INVALID", "API KEY NOT VALID", "UNAUTHENTICATED")):
            raise

if llm is None:
    raise RuntimeError("No Gemini Flash model responded. Check API access/quota in Google AI Studio or retry when service is available.")

# Preserve the selected model and use alternatives if later requests fail.
from langchain_core.runnables import RunnableLambda
available_llms = [llm] + [m for m in candidates if m is not llm]

def invoke_gemini(messages):
    last_error = None
    for model in available_llms:
        try:
            return model.invoke(messages)
        except Exception as exc:
            if not is_temporary_error(exc) and not any(x in str(exc).upper() for x in ("404", "NOT_FOUND")):
                raise
            last_error = exc
    raise RuntimeError("All available Gemini models failed; check service availability or quota.") from last_error

gemini_with_fallback = RunnableLambda(invoke_gemini)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser

SYSTEM = """ROLE:   You are the friendly assistant for Sweet Crumbs Home Bakery.
FACTS:  Plum cake Rs 500. Chocolate cake Rs 650. Cupcake Rs 80 each.
        Orders need 24 hours' notice. All cakes have eggless versions.
RULES:  Never promise a delivery time or place. If it isn't in FACTS,
        say "I'll check with the owner." Work out totals carefully.
FORMAT: Under 50 words, friendly, English or Telugu matching the customer."""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM),
    MessagesPlaceholder("history"),          # the chat so far goes here
    ("human", "{question}"),
])
chain = prompt | gemini_with_fallback | StrOutputParser()

history = []
def ask(question: str) -> str:
    answer = chain.invoke({"history": history, "question": question})
    history.extend([("human", question), ("ai", answer)])
    return answer

print(ask("Hi, I'm Ravi. Do you make eggless cakes?"))
print(ask("What's my name?"))

In [ ]:
history

In [ ]:
%%writefile tests.csv
question,must_include
How much is a plum cake?,500
Do you make eggless cakes?,eggless
Can I get a cake today evening?,24 hours
Do you deliver to Kakinada?,check with the owner
What is the price of 6 cupcakes?,480

In [ ]:
import csv

def run_tests(path: str = "tests.csv") -> None:
    rows = list(csv.DictReader(open(path)))
    passed = 0
    for row in rows:
        answer = chain.invoke({"history": [], "question": row["question"]})
        ok = row["must_include"].lower() in answer.lower()
        passed += ok
        print("PASS" if ok else "FAIL", "|", row["question"], "→", answer[:70])
    print(f"\n{passed}/{len(rows)} passed ({passed / len(rows):.0%})")



In [ ]:
run_tests()

In [ ]:
from langchain.agents import create_agent

# tools come tomorrow
last_error = None
for model in available_llms:
    try:
        agent_order = create_agent(model=model, tools=[], system_prompt=SYSTEM)
        result_order = agent_order.invoke({"messages": [{"role": "user", "content": "Do you make eggless cakes?"}]})
        print(StrOutputParser().invoke(result_order["messages"][-1]))
        break
    except Exception as exc:
        if not is_temporary_error(exc) and not any(x in str(exc).upper() for x in ("404", "NOT_FOUND")):
            raise
        last_error = exc
else:
    raise RuntimeError("All available Gemini models failed during the agent test.") from last_error